# UAP Video Frame Analysis

This notebook creates a reproducible workflow for processing UAP (*unidentified anomalous phenomena*) videos. The main function receives a video file, reads its FPS, exports every frame as an image, and compares the apparent target motion with the global scene/camera motion.

The final classification is heuristic: it points to the most likely cause of a target disappearing from the scene, using evidence such as residual object speed, residual acceleration, camera motion change, and target proximity to the frame edge.

## Dependencies

If needed, install the libraries below in the notebook environment:

```bash
pip install opencv-python numpy pandas matplotlib
```

In [ ]:
from __future__ import annotations

from pathlib import Path
import math
from typing import Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)

try:
    import cv2
except ImportError as exc:
    raise ImportError(
        "Install the dependencies with: pip install opencv-python numpy pandas matplotlib"
    ) from exc


def _safe_fps(cap: cv2.VideoCapture, fallback: float = 30.0) -> float:
    fps = float(cap.get(cv2.CAP_PROP_FPS) or 0.0)
    if not np.isfinite(fps) or fps <= 0:
        return fallback
    return fps


def _validate_box(box: tuple[int, int, int, int] | None, width: int, height: int, name: str):
    if box is None:
        return None
    x, y, w, h = [int(v) for v in box]
    if w <= 0 or h <= 0:
        raise ValueError(f"{name} must have positive width and height: {box}")
    if x < 0 or y < 0 or x + w > width or y + h > height:
        raise ValueError(f"{name} is outside the {width}x{height} frame: {box}")
    return (x, y, w, h)


def _crop(frame: np.ndarray, roi: tuple[int, int, int, int] | None):
    if roi is None:
        return frame, (0, 0)
    x, y, w, h = roi
    return frame[y : y + h, x : x + w], (x, y)


def _to_gray(frame: np.ndarray) -> np.ndarray:
    if frame.ndim == 3:
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    else:
        gray = frame.copy()
    gray = cv2.GaussianBlur(gray, (3, 3), 0)
    return gray


def _identity_affine() -> np.ndarray:
    return np.array([[1.0, 0.0, 0.0], [0.0, 1.0, 0.0]], dtype=np.float32)


def _transform_point(point: np.ndarray, affine: np.ndarray) -> np.ndarray:
    point = np.asarray(point, dtype=np.float32).reshape(1, 1, 2)
    return cv2.transform(point, affine).reshape(2).astype(float)


def _edge_distance(point: np.ndarray, width: int, height: int) -> float:
    x, y = float(point[0]), float(point[1])
    return float(min(x, y, width - 1 - x, height - 1 - y))


def _estimate_global_motion(
    prev_gray: np.ndarray,
    curr_gray: np.ndarray,
    max_features: int = 2500,
    keep_percent: float = 0.25,
    ransac_reproj_threshold: float = 3.0,
) -> tuple[np.ndarray, dict[str, Any]]:
    affine = _identity_affine()
    metrics = {
        "camera_dx_px": 0.0,
        "camera_dy_px": 0.0,
        "camera_translation_px": 0.0,
        "camera_scale": 1.0,
        "camera_rotation_deg": 0.0,
        "camera_match_count": 0,
        "camera_inlier_ratio": 0.0,
        "camera_median_residual_px": np.nan,
        "camera_motion_ok": False,
    }

    orb = cv2.ORB_create(nfeatures=max_features, fastThreshold=7)
    kp_prev, desc_prev = orb.detectAndCompute(prev_gray, None)
    kp_curr, desc_curr = orb.detectAndCompute(curr_gray, None)
    if desc_prev is None or desc_curr is None or len(kp_prev) < 8 or len(kp_curr) < 8:
        return affine, metrics

    matcher = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)
    matches = matcher.match(desc_prev, desc_curr)
    if len(matches) < 8:
        return affine, metrics

    matches = sorted(matches, key=lambda m: m.distance)
    keep = max(8, int(len(matches) * keep_percent))
    matches = matches[:keep]

    prev_pts = np.float32([kp_prev[m.queryIdx].pt for m in matches])
    curr_pts = np.float32([kp_curr[m.trainIdx].pt for m in matches])
    affine_est, inliers = cv2.estimateAffinePartial2D(
        prev_pts,
        curr_pts,
        method=cv2.RANSAC,
        ransacReprojThreshold=ransac_reproj_threshold,
        maxIters=2000,
        confidence=0.99,
    )
    if affine_est is None:
        metrics["camera_match_count"] = len(matches)
        return affine, metrics

    affine = affine_est.astype(np.float32)
    inlier_mask = inliers.ravel().astype(bool) if inliers is not None else np.ones(len(matches), dtype=bool)
    transformed = cv2.transform(prev_pts.reshape(-1, 1, 2), affine).reshape(-1, 2)
    residuals = np.linalg.norm(transformed - curr_pts, axis=1)
    residuals_inliers = residuals[inlier_mask] if np.any(inlier_mask) else residuals

    a, b, tx = affine[0]
    c, d, ty = affine[1]
    metrics.update(
        {
            "camera_dx_px": float(tx),
            "camera_dy_px": float(ty),
            "camera_translation_px": float(math.hypot(tx, ty)),
            "camera_scale": float(math.sqrt(max(a * a + c * c, 0.0))),
            "camera_rotation_deg": float(math.degrees(math.atan2(c, a))),
            "camera_match_count": int(len(matches)),
            "camera_inlier_ratio": float(np.mean(inlier_mask)),
            "camera_median_residual_px": float(np.median(residuals_inliers)),
            "camera_motion_ok": True,
        }
    )
    return affine, metrics


def _detect_residual_blobs(
    prev_gray: np.ndarray,
    curr_gray: np.ndarray,
    affine: np.ndarray,
    min_blob_area: int = 4,
    max_blob_area_ratio: float = 0.01,
    diff_threshold: int | None = None,
) -> list[dict[str, Any]]:
    height, width = curr_gray.shape[:2]
    warped_prev = cv2.warpAffine(
        prev_gray,
        affine,
        (width, height),
        flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_REPLICATE,
    )
    diff = cv2.absdiff(warped_prev, curr_gray)
    diff_blur = cv2.GaussianBlur(diff, (5, 5), 0)

    if diff_threshold is None:
        otsu_value, mask = cv2.threshold(diff_blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        if otsu_value < 10:
            _, mask = cv2.threshold(diff_blur, 10, 255, cv2.THRESH_BINARY)
    else:
        _, mask = cv2.threshold(diff_blur, int(diff_threshold), 255, cv2.THRESH_BINARY)

    kernel = np.ones((3, 3), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=1)
    mask = cv2.dilate(mask, kernel, iterations=1)

    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    max_blob_area = float(width * height * max_blob_area_ratio)
    candidates: list[dict[str, Any]] = []
    for contour in contours:
        area = float(cv2.contourArea(contour))
        if area < min_blob_area or area > max_blob_area:
            continue
        x, y, w, h = cv2.boundingRect(contour)
        moments = cv2.moments(contour)
        if moments["m00"]:
            cx = moments["m10"] / moments["m00"]
            cy = moments["m01"] / moments["m00"]
        else:
            cx = x + w / 2
            cy = y + h / 2
        patch = diff[y : y + h, x : x + w]
        score = area * (float(np.mean(patch)) + float(np.max(patch)))
        candidates.append(
            {
                "cx": float(cx),
                "cy": float(cy),
                "x": int(x),
                "y": int(y),
                "w": int(w),
                "h": int(h),
                "area": area,
                "score": float(score),
            }
        )
    return sorted(candidates, key=lambda c: c["score"], reverse=True)


def _choose_candidate(
    candidates: list[dict[str, Any]],
    predicted_center: np.ndarray | None,
    max_distance_px: float,
) -> tuple[dict[str, Any] | None, float]:
    if not candidates:
        return None, np.nan
    if predicted_center is None:
        return candidates[0], np.nan
    ranked = []
    for candidate in candidates:
        center = np.array([candidate["cx"], candidate["cy"]], dtype=float)
        distance = float(np.linalg.norm(center - predicted_center))
        ranked.append((distance, -candidate["score"], candidate))
    ranked.sort(key=lambda item: (item[0], item[1]))
    distance, _, candidate = ranked[0]
    if distance <= max_distance_px:
        return candidate, distance
    return None, distance


def _median_or_nan(values: pd.Series) -> float:
    values = pd.to_numeric(values, errors="coerce").dropna()
    if values.empty:
        return float("nan")
    return float(values.median())


def _ratio(after: float, before: float) -> float:
    if not np.isfinite(after) or not np.isfinite(before):
        return float("nan")
    return float(after / max(abs(before), 1e-6))


def _classify_disappearance(
    track: pd.DataFrame,
    missing_start_pos: int,
    missing_end_pos: int,
    edge_margin_px: float,
    window: int = 8,
) -> dict[str, Any]:
    before = track.iloc[max(0, missing_start_pos - window) : missing_start_pos].copy()
    after = track.iloc[missing_start_pos : min(len(track), missing_start_pos + window)].copy()
    detected_before = before[before["detected"]]

    if detected_before.empty:
        return {}

    last = detected_before.iloc[-1]
    early = detected_before.iloc[: max(1, len(detected_before) // 2)]
    late = detected_before.iloc[max(0, len(detected_before) - max(2, len(detected_before) // 2)) :]

    camera_before = _median_or_nan(before["camera_speed_px_s"])
    camera_after = _median_or_nan(after["camera_speed_px_s"])
    camera_change_ratio = _ratio(camera_after, camera_before)

    object_speed_early = _median_or_nan(early["uap_residual_speed_px_s"])
    object_speed_late = _median_or_nan(late["uap_residual_speed_px_s"])
    object_speed_ratio = _ratio(object_speed_late, object_speed_early)
    object_accel_late = _median_or_nan(late["uap_residual_accel_px_s2"])

    near_edge = bool(float(last.get("distance_to_edge_px", np.inf)) <= edge_margin_px)
    camera_stopped = bool(np.isfinite(camera_before) and np.isfinite(camera_after) and camera_before > 10 and camera_after < camera_before * 0.45)
    camera_moved = bool(np.isfinite(camera_before) and np.isfinite(camera_after) and camera_after > max(10, camera_before * 1.7))
    object_accelerated = bool(
        (np.isfinite(object_speed_ratio) and object_speed_ratio > 1.6 and object_speed_late > 8)
        or (np.isfinite(object_accel_late) and object_accel_late > 25)
    )
    object_decelerated = bool(
        (np.isfinite(object_speed_ratio) and object_speed_ratio < 0.55 and object_speed_early > 8)
        or (np.isfinite(object_accel_late) and object_accel_late < -25)
    )

    if camera_stopped and near_edge:
        classification = "camera_stopped_tracking"
        confidence = 0.78
        reason = "The global camera motion dropped around the disappearance and the target was near the frame edge."
    elif camera_moved and not object_accelerated:
        classification = "camera_moved"
        confidence = 0.72
        reason = "The global scene motion increased at the disappearance, without strong evidence of target residual acceleration."
    elif object_accelerated and near_edge:
        classification = "uap_accelerated_out_of_scene"
        confidence = 0.76
        reason = "The target residual speed increased and the last detection was near the frame edge."
    elif object_accelerated:
        classification = "uap_accelerated"
        confidence = 0.64
        reason = "The target residual speed/acceleration increased after compensating for camera motion."
    elif object_decelerated:
        classification = "uap_decelerated_or_lost_contrast"
        confidence = 0.58
        reason = "Residual speed dropped before the disappearance; this may indicate deceleration or loss of contrast/detection."
    elif camera_moved:
        classification = "camera_moved"
        confidence = 0.62
        reason = "There is a relevant change in global camera motion near the disappearance."
    else:
        classification = "undetermined"
        confidence = 0.35
        reason = "The camera-motion and target-residual evidence did not separate a dominant cause."

    return {
        "missing_start_frame": int(track.iloc[missing_start_pos]["frame_index"]),
        "missing_start_time_s": float(track.iloc[missing_start_pos]["time_s"]),
        "last_seen_frame": int(last["frame_index"]),
        "last_seen_time_s": float(last["time_s"]),
        "missing_frames": int(missing_end_pos - missing_start_pos),
        "classification": classification,
        "confidence": confidence,
        "reason": reason,
        "camera_speed_before_px_s": camera_before,
        "camera_speed_after_px_s": camera_after,
        "camera_change_ratio": camera_change_ratio,
        "uap_speed_early_px_s": object_speed_early,
        "uap_speed_late_px_s": object_speed_late,
        "uap_speed_ratio": object_speed_ratio,
        "uap_accel_late_px_s2": object_accel_late,
        "last_distance_to_edge_px": float(last.get("distance_to_edge_px", np.nan)),
    }


def _find_disappearance_events(
    track: pd.DataFrame,
    disappearance_patience: int,
    edge_margin_px: float,
) -> pd.DataFrame:
    if track.empty or "detected" not in track:
        return pd.DataFrame()
    detected = track["detected"].fillna(False).to_numpy(dtype=bool)
    events = []
    pos = 1
    while pos < len(detected):
        if detected[pos - 1] and not detected[pos]:
            start = pos
            while pos < len(detected) and not detected[pos]:
                pos += 1
            end = pos
            if end - start >= disappearance_patience:
                event = _classify_disappearance(track, start, end, edge_margin_px=edge_margin_px)
                if event:
                    events.append(event)
        else:
            pos += 1
    return pd.DataFrame(events)


def analyze_uap_video(
    video_path: str | Path,
    output_dir: str | Path | None = None,
    roi: tuple[int, int, int, int] | None = None,
    object_bbox: tuple[int, int, int, int] | None = None,
    frame_step: int = 1,
    image_ext: str = "png",
    disappearance_patience: int = 5,
    min_blob_area: int = 4,
    max_blob_area_ratio: float = 0.01,
    diff_threshold: int | None = None,
    max_tracking_jump_px: float | None = None,
    verbose: bool = True,
) -> dict[str, Any]:
    """Extract video frames and classify disappearances of a possible UAP.

    Main parameters
    ---------------------
    video_path:
        Path to the video file.
    output_dir:
        Output folder. If omitted, a folder is created next to the video.
    roi:
        Region of interest in the full frame, as (x, y, width, height).
        Use this to limit analysis to the area where the UAP appears.
    object_bbox:
        Approximate initial UAP box in the full frame, as (x, y, width, height).
        This helps a lot when there are other moving objects in the video.
    frame_step:
        Process/export 1 out of every N frames. Keep 1 to export all frames.

    Returns
    -------
    dict with metadata, frame/track/event DataFrames, and the frame output folder.
    """
    video_path = Path(video_path).expanduser().resolve()
    if not video_path.exists():
        raise FileNotFoundError(f"Video not found: {video_path}")
    if frame_step < 1:
        raise ValueError("frame_step must be >= 1")

    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise ValueError(f"Could not open video: {video_path}")

    fps = _safe_fps(cap)
    frame_count_nominal = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH) or 0)
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT) or 0)
    duration_s = frame_count_nominal / fps if fps else np.nan

    roi = _validate_box(roi, width, height, "roi")
    object_bbox = _validate_box(object_bbox, width, height, "object_bbox")

    if output_dir is None:
        output_dir = video_path.parent / f"{video_path.stem}_uap_analysis"
    output_dir = Path(output_dir).expanduser().resolve()
    frames_dir = output_dir / "frames"
    frames_dir.mkdir(parents=True, exist_ok=True)

    roi_width = roi[2] if roi else width
    roi_height = roi[3] if roi else height
    edge_margin_px = 0.08 * math.hypot(roi_width, roi_height)
    if max_tracking_jump_px is None:
        max_tracking_jump_px = max(12.0, 0.06 * math.hypot(roi_width, roi_height))

    frame_rows: list[dict[str, Any]] = []
    track_rows: list[dict[str, Any]] = []

    prev_gray: np.ndarray | None = None
    prev_frame_index: int | None = None
    prev_time_s: float | None = None
    last_center: np.ndarray | None = None
    residual_velocity = np.array([0.0, 0.0], dtype=float)
    last_residual_speed = np.nan
    started_tracking = False

    read_index = 0
    saved_count = 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break

        frame_index = read_index
        read_index += 1
        if frame_index % frame_step != 0:
            continue

        time_s = frame_index / fps
        frame_path = frames_dir / f"frame_{frame_index:06d}.{image_ext}"
        if not cv2.imwrite(str(frame_path), frame):
            raise IOError(f"Failed to save frame: {frame_path}")

        frame_rows.append(
            {
                "frame_index": frame_index,
                "time_s": time_s,
                "path": str(frame_path),
                "width": width,
                "height": height,
            }
        )
        saved_count += 1

        cropped, origin = _crop(frame, roi)
        gray = _to_gray(cropped)

        if not started_tracking and object_bbox is not None:
            bx, by, bw, bh = object_bbox
            ox, oy = origin
            initial_center = np.array([bx + bw / 2 - ox, by + bh / 2 - oy], dtype=float)
            if 0 <= initial_center[0] < roi_width and 0 <= initial_center[1] < roi_height:
                last_center = initial_center
                started_tracking = True

        if prev_gray is None:
            prev_gray = gray
            prev_frame_index = frame_index
            prev_time_s = time_s
            continue

        dt = max(time_s - float(prev_time_s), 1.0 / fps)
        affine, camera_metrics = _estimate_global_motion(prev_gray, gray)
        candidates = _detect_residual_blobs(
            prev_gray,
            gray,
            affine,
            min_blob_area=min_blob_area,
            max_blob_area_ratio=max_blob_area_ratio,
            diff_threshold=diff_threshold,
        )

        camera_pred = None
        predicted_center = None
        if last_center is not None:
            camera_pred = _transform_point(last_center, affine)
            predicted_center = camera_pred + residual_velocity * dt

        chosen, nearest_distance = _choose_candidate(candidates, predicted_center, float(max_tracking_jump_px))
        if chosen is not None:
            measured_center = np.array([chosen["cx"], chosen["cy"]], dtype=float)
            if camera_pred is not None:
                residual_displacement = measured_center - camera_pred
                residual_speed = float(np.linalg.norm(residual_displacement) / dt)
                residual_velocity = residual_displacement / dt
                residual_accel = (
                    float((residual_speed - last_residual_speed) / dt)
                    if np.isfinite(last_residual_speed)
                    else np.nan
                )
            else:
                residual_displacement = np.array([np.nan, np.nan])
                residual_speed = np.nan
                residual_accel = np.nan
                residual_velocity = np.array([0.0, 0.0], dtype=float)

            last_center = measured_center
            last_residual_speed = residual_speed
            started_tracking = True
            detected = True
            distance_to_edge = _edge_distance(measured_center, roi_width, roi_height)
            uap_x_px = float(measured_center[0] + origin[0])
            uap_y_px = float(measured_center[1] + origin[1])
            predicted_x_px = float(predicted_center[0] + origin[0]) if predicted_center is not None else np.nan
            predicted_y_px = float(predicted_center[1] + origin[1]) if predicted_center is not None else np.nan
            blob_area = float(chosen["area"])
            blob_score = float(chosen["score"])
        else:
            detected = False
            if predicted_center is not None:
                last_center = predicted_center
            distance_to_edge = _edge_distance(last_center, roi_width, roi_height) if last_center is not None else np.nan
            residual_speed = np.nan
            residual_accel = np.nan
            uap_x_px = np.nan
            uap_y_px = np.nan
            predicted_x_px = float(predicted_center[0] + origin[0]) if predicted_center is not None else np.nan
            predicted_y_px = float(predicted_center[1] + origin[1]) if predicted_center is not None else np.nan
            blob_area = np.nan
            blob_score = np.nan

        row = {
            "frame_index": frame_index,
            "prev_frame_index": int(prev_frame_index),
            "time_s": time_s,
            "dt_s": dt,
            "detected": detected,
            "candidate_count": len(candidates),
            "nearest_candidate_distance_px": nearest_distance,
            "uap_x_px": uap_x_px,
            "uap_y_px": uap_y_px,
            "predicted_uap_x_px": predicted_x_px,
            "predicted_uap_y_px": predicted_y_px,
            "uap_blob_area_px": blob_area,
            "uap_blob_score": blob_score,
            "uap_residual_speed_px_s": residual_speed,
            "uap_residual_accel_px_s2": residual_accel,
            "distance_to_edge_px": distance_to_edge,
            **camera_metrics,
        }
        row["camera_speed_px_s"] = row["camera_translation_px"] / dt
        track_rows.append(row)

        prev_gray = gray
        prev_frame_index = frame_index
        prev_time_s = time_s

    cap.release()

    frames = pd.DataFrame(frame_rows)
    track = pd.DataFrame(track_rows)
    events = _find_disappearance_events(track, disappearance_patience, edge_margin_px=edge_margin_px)

    metadata = {
        "video_path": str(video_path),
        "fps": fps,
        "nominal_frame_count": frame_count_nominal,
        "processed_frame_count": len(frames),
        "width": width,
        "height": height,
        "duration_s": duration_s,
        "frame_step": frame_step,
        "frames_dir": str(frames_dir),
        "roi": roi,
        "object_bbox": object_bbox,
        "edge_margin_px": edge_margin_px,
        "max_tracking_jump_px": max_tracking_jump_px,
    }

    if verbose:
        print(f"Detected FPS: {fps:.3f}")
        print(f"Exported frames: {len(frames)} to {frames_dir}")
        if events.empty:
            print("No persistent disappearance was detected with the current parameters.")
        else:
            display_cols = [
                "last_seen_frame",
                "missing_start_frame",
                "classification",
                "confidence",
                "reason",
            ]
            display(events[display_cols])

    return {
        "metadata": metadata,
        "frames": frames,
        "track": track,
        "events": events,
        "frames_dir": frames_dir,
    }

def plot_diagnostics(result: dict[str, Any]):
    """Plot camera motion, UAP residual speed, and residual acceleration."""
    track = result["track"]
    events = result["events"]
    if track.empty:
        raise ValueError("There is no track data to plot.")

    fig, axes = plt.subplots(3, 1, figsize=(13, 8), sharex=True)
    axes[0].plot(track["time_s"], track["camera_speed_px_s"], color="tab:blue", label="camera")
    axes[0].set_ylabel("Camera\n(px/s)")
    axes[0].grid(True, alpha=0.25)

    detected = track[track["detected"]]
    axes[1].plot(
        detected["time_s"],
        detected["uap_residual_speed_px_s"],
        color="tab:orange",
        marker="o",
        ms=3,
        label="UAP residual",
    )
    axes[1].set_ylabel("UAP residual\n(px/s)")
    axes[1].grid(True, alpha=0.25)

    axes[2].plot(
        detected["time_s"],
        detected["uap_residual_accel_px_s2"],
        color="tab:green",
        marker="o",
        ms=3,
        label="residual acceleration",
    )
    axes[2].axhline(0, color="black", lw=0.8, alpha=0.4)
    axes[2].set_ylabel("Acceleration\n(px/s2)")
    axes[2].set_xlabel("Time (s)")
    axes[2].grid(True, alpha=0.25)

    if not events.empty:
        for _, event in events.iterrows():
            for ax in axes:
                ax.axvline(event["missing_start_time_s"], color="crimson", ls="--", alpha=0.7)
            axes[0].annotate(
                event["classification"],
                xy=(event["missing_start_time_s"], axes[0].get_ylim()[1]),
                xytext=(5, -15),
                textcoords="offset points",
                color="crimson",
                fontsize=9,
                va="top",
            )

    fig.tight_layout()
    return fig


def plot_axis_diagnostics(result: dict[str, Any]):
    """Plot camera X/Y motion and UAP X/Y position to diagnose axis-specific behavior."""
    track = result["track"]
    events = result["events"]
    if track.empty:
        raise ValueError("There is no track data to plot.")

    detected = track[track["detected"]]
    fig, axes = plt.subplots(5, 1, figsize=(13, 11), sharex=True)

    axes[0].plot(track["time_s"], track["camera_dx_px"], color="tab:blue", label="camera dx")
    axes[0].axhline(0, color="black", lw=0.8, alpha=0.35)
    axes[0].set_ylabel("Camera dx\n(px/step)")
    axes[0].grid(True, alpha=0.25)

    axes[1].plot(track["time_s"], track["camera_dy_px"], color="tab:cyan", label="camera dy")
    axes[1].axhline(0, color="black", lw=0.8, alpha=0.35)
    axes[1].set_ylabel("Camera dy\n(px/step)")
    axes[1].grid(True, alpha=0.25)

    axes[2].plot(
        detected["time_s"],
        detected["uap_x_px"],
        color="tab:orange",
        marker="o",
        ms=3,
        label="detected UAP x",
    )
    axes[2].plot(
        track["time_s"],
        track["predicted_uap_x_px"],
        color="tab:orange",
        alpha=0.35,
        ls="--",
        label="predicted UAP x",
    )
    axes[2].set_ylabel("UAP x\n(px)")
    axes[2].grid(True, alpha=0.25)

    axes[3].plot(
        detected["time_s"],
        detected["uap_y_px"],
        color="tab:green",
        marker="o",
        ms=3,
        label="detected UAP y",
    )
    axes[3].plot(
        track["time_s"],
        track["predicted_uap_y_px"],
        color="tab:green",
        alpha=0.35,
        ls="--",
        label="predicted UAP y",
    )
    axes[3].set_ylabel("UAP y\n(px)")
    axes[3].grid(True, alpha=0.25)

    axes[4].plot(
        detected["time_s"],
        detected["uap_residual_speed_px_s"],
        color="tab:red",
        marker="o",
        ms=3,
        label="UAP residual speed",
    )
    axes[4].set_ylabel("Residual speed\n(px/s)")
    axes[4].set_xlabel("Time (s)")
    axes[4].grid(True, alpha=0.25)

    for ax in axes:
        ax.legend(loc="upper right", fontsize=8)

    if not events.empty:
        for _, event in events.iterrows():
            for ax in axes:
                ax.axvline(event["missing_start_time_s"], color="crimson", ls="--", alpha=0.7)
            axes[0].annotate(
                event["classification"],
                xy=(event["missing_start_time_s"], axes[0].get_ylim()[1]),
                xytext=(5, -15),
                textcoords="offset points",
                color="crimson",
                fontsize=9,
                va="top",
            )

    fig.tight_layout()
    return fig

## How to Use

The simplest call uses only the video path. For videos with many moving objects, also provide a `roi` and/or an approximate `object_bbox` in the first frame where the UAP appears.

- `roi=(x, y, width, height)` limits analysis to part of the frame.
- `object_bbox=(x, y, width, height)` initializes tracking on the correct target.
- `disappearance_patience` defines how many frames without detection count as a real disappearance.

In [ ]:
result = analyze_uap_video(
    "/content/hyperaccel1.mp4",
    output_dir="/content/uap_analysis_output",
    roi=None,                 # example: (100, 80, 900, 500)
    object_bbox=None,         # example: (460, 220, 18, 18)
    frame_step=1,
    disappearance_patience=5,
)

result["metadata"]
result["events"]
plot_diagnostics(result)


## Interpreting the Results

The `result["events"]` DataFrame summarizes each persistent disappearance. Possible classifications are:

- `uap_accelerated_out_of_scene`: the target accelerated after camera compensation and was near the frame edge.
- `uap_accelerated`: residual speed/acceleration increased, but without clear evidence of leaving through the edge.
- `uap_decelerated_or_lost_contrast`: residual target speed dropped before disappearance; this may be deceleration or contrast loss.
- `camera_moved`: global scene motion better explains the disappearance.
- `camera_stopped_tracking`: the camera reduced tracking motion and the target was near the frame edge.
- `undetermined`: the evidence does not separate a dominant cause.

For a stronger analysis, tune `roi`, `object_bbox`, `min_blob_area`, `diff_threshold`, and `disappearance_patience`, then compare the charts generated by `plot_diagnostics` and `plot_axis_diagnostics`.

`plot_axis_diagnostics(result)` is useful when the camera and target move differently by axis. It plots camera `dx`, camera `dy`, detected/predicted UAP `x`, detected/predicted UAP `y`, and residual UAP speed. Use it to check cases where the camera stops tracking horizontally while still moving vertically.